In [6]:
# ============================================================
# Evaluation: CLIP vs. Gold Standard (Indoor/Outdoor)
# ============================================================

import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score,
    classification_report,
    confusion_matrix
)

# ── Pfade ────────────────────────────────────────────────────
INPUT_CSV = r'C:\Users\Lukas\masterthesis\thesis_2026\data\Annotation_Indoor_Outdoor\annotations_indoor_outdoor.csv'

OUTPUT_DIR = r'C:\Users\Lukas\masterthesis\thesis_2026\data\Annotation_Indoor_Outdoor\evaluation_indoor_outdoor'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Daten laden ──────────────────────────────────────────────
df = pd.read_csv(INPUT_CSV)

print(f'Bilder geladen: {len(df)}')
print(df.head())

true = df['gold_standard']
pred = df['annotator_CLIP']

categories = sorted(df['gold_standard'].dropna().unique())

print(f'\nKategorien: {categories}')

# ============================================================
# Agreement / Disagreement Analysis
# ============================================================

df['agreement'] = (
    df['gold_standard'] == df['annotator_CLIP']
)

agreements = df[df['agreement']]
disagreements = df[~df['agreement']]

agreement_rate = len(agreements) / len(df)
disagreement_rate = len(disagreements) / len(df)

print("\n==============================")
print("AGREEMENT ANALYSIS")
print("==============================")
print(f"Agreements       : {len(agreements)}")
print(f"Disagreements    : {len(disagreements)}")
print(f"Agreement Rate   : {agreement_rate:.1%}")
print(f"Disagreement Rate: {disagreement_rate:.1%}")

# CSV-Dateien speichern
agreements.to_csv(
    os.path.join(OUTPUT_DIR, "agreements_gold_vs_clip.csv"),
    index=False
)

disagreements.to_csv(
    os.path.join(OUTPUT_DIR, "disagreements_gold_vs_clip.csv"),
    index=False
)

print("✅ agreements_gold_vs_clip.csv saved")
print("✅ disagreements_gold_vs_clip.csv saved")

# Fehlklassifikationen anzeigen
if len(disagreements) > 0:
    print("\nMisclassified Images:")
    cols = [c for c in ['filename', 'gold_standard', 'annotator_CLIP'] if c in df.columns]
    print(disagreements[cols])

# ============================================================
# Evaluation Metrics
# ============================================================

accuracy = accuracy_score(true, pred)
precision = precision_score(true, pred, average='weighted')
recall = recall_score(true, pred, average='weighted')
f1 = f1_score(true, pred, average='weighted')
kappa = cohen_kappa_score(true, pred)

print("\n==============================")
print("EVALUATION RESULTS")
print("==============================")
print(f"Accuracy      : {accuracy:.4f}")
print(f"Precision     : {precision:.4f}")
print(f"Recall        : {recall:.4f}")
print(f"F1-Score      : {f1:.4f}")
print(f"Cohen's Kappa : {kappa:.4f}")

# ============================================================
# Classification Report
# ============================================================

report = classification_report(
    true,
    pred,
    digits=4
)

print("\nClassification Report:")
print(report)

# ============================================================
# Save Metrics
# ============================================================

metrics_df = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-Score",
        "Cohen's Kappa",
        "Agreements",
        "Disagreements",
        "Agreement Rate",
        "Disagreement Rate"
    ],
    "Value": [
        accuracy,
        precision,
        recall,
        f1,
        kappa,
        len(agreements),
        len(disagreements),
        agreement_rate,
        disagreement_rate
    ]
})

metrics_df.to_csv(
    os.path.join(OUTPUT_DIR, "evaluation_metrics.csv"),
    index=False
)

# ============================================================
# Confusion Matrix
# ============================================================

cm = confusion_matrix(
    true,
    pred,
    labels=categories
)

cm_df = pd.DataFrame(
    cm,
    index=[f"True_{c}" for c in categories],
    columns=[f"Pred_{c}" for c in categories]
)

cm_df.to_csv(
    os.path.join(OUTPUT_DIR, "confusion_matrix.csv")
)

plt.figure(figsize=(5, 4))

sns.heatmap(
    cm_df,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title("Confusion Matrix")
plt.tight_layout()

plt.savefig(
    os.path.join(OUTPUT_DIR, "confusion_matrix.png"),
    dpi=300
)

plt.close()

print("\n✅ evaluation_metrics.csv saved")
print("✅ confusion_matrix.csv saved")
print("✅ confusion_matrix.png saved")
print(f"\nAlle Ergebnisse gespeichert unter:\n{OUTPUT_DIR}")

Bilder geladen: 300
          filename annotator_NB annotator_LE gold_standard annotator_CLIP
0  image_10020.jpg       indoor       indoor        indoor        outdoor
1  image_10026.jpg      outdoor      outdoor       outdoor        outdoor
2   image_1009.jpg      outdoor      outdoor       outdoor        outdoor
3  image_10104.jpg      outdoor      outdoor       outdoor        outdoor
4  image_10134.jpg      outdoor      outdoor       outdoor        outdoor

Kategorien: ['indoor', 'outdoor']

AGREEMENT ANALYSIS
Agreements       : 289
Disagreements    : 11
Agreement Rate   : 96.3%
Disagreement Rate: 3.7%
✅ agreements_gold_vs_clip.csv saved
✅ disagreements_gold_vs_clip.csv saved

Misclassified Images:
            filename gold_standard annotator_CLIP
0    image_10020.jpg        indoor        outdoor
97     image_198.jpg       outdoor         indoor
113   image_2516.jpg        indoor        outdoor
128   image_2850.jpg        indoor        outdoor
130   image_2870.jpg        indoor     